# Landfall Detection Tool
This notebook is the landfall detection algorithm run for a given dataset with AR masks as the input.

In [1]:
import xarray as xa
import itertools
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import time
import pandas as pd
import gc

In [2]:
# When I originally made this, I used a PBS Cluster to process all the data quickly.
# You can probably get away with a LocalCluster system if you request enough memory
#   and update the module import as necessary.

cluster = PBSCluster(
    account='WYOM0169',
    queue='main',
    cores=4,
    memory='64GB',
    walltime='08:00:00',
    job_extra_directives=['-l select=1:ncpus:4:mem=64GB', '-o /glade/u/home/tcorrie/dask_worker_log/', '-e /glade/u/home/tcorrie/dask_worker_log/'],
    )
cluster.scale(64)
client = Client(cluster)

In [4]:
print(client)

<Client: 'tcp://128.117.208.218:41669' processes=64 threads=64, memory=0.93 TiB>


In [6]:
# process_timestep is how each timestep is processed for landfall detection.
# We are checking for landfall overall AND in each region.

def process_timestep(tstep, ardt, model, member):
    ARmask = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARmasks/{ardt}/ARmask.{model}.{member}.nc', use_cftime=True) # Get the ouputted AR mask for the inputted ARDT
    land = xa.open_dataset('/glade/derecho/scratch/tcorrie/regrids/landmask_regridded.nc') # Get the regridded landmask
    coast = xa.where(land['LANDMASK'] >= 0.5, 1, 0).idxmax(dim='lon') # Define the coast points
    coastline = coast.sel(lat=slice(32.5,48.51)) # Set coast bounds
    coast_lats = coastline.lat.data
    coast_lons = coastline.data
    # Overlay the AR mask with the 1-D coast markers
    ARmaskcoast = [ARmask.isel(time=tstep).sel(lat=lat, lon=lon).ARmask.values.astype('int').item() for lat, lon in zip(coast_lats, coast_lons)]
    # Check for minimum WUS landfall (1 deg, 10 pixels) in range of 32.5 to 48.5
    landfall_widths = [(key, len(list(group))) for key, group in itertools.groupby(ARmaskcoast) if key == 1]
    landfall_extent = [lw[1]/10 for lw in landfall_widths]

    if max(landfall_extent, default=0) < 1.0: # If landfall extent is less that 1 degree in latitude
        return (0, 0, 0, 0, 0, ARmaskcoast) # Order: WUS, SCA, NCA, OR, WA. We can end program here as there is no longer a region to check.

    def check_region(lat_min, lat_max): # Sub-function to check which region the landfalling AR has affected.
        cl = coast.sel(lat=slice(lat_min, lat_max))
        amc = [ARmask.isel(time=tstep).sel(lat=lat, lon=lon).ARmask.values.astype('int').item() for lat, lon in zip(cl.lat.data, cl.data)]
        widths = [(key, len(list(group))) for key, group in itertools.groupby(amc) if key == 1]
        extents = [lw[1]/10 for lw in widths]
        return 1 if max(extents, default=0) >= 0.5 else 0 # For an AR mask to be in a given region we require a minimum landfall of half the minimum overall 

    # Note that these latitude bounds are only for the coast and may not be reflective of the entire state itself.
    landfall_sca = check_region(32.5, 37.21) # SCA/SoCal
    landfall_nca = check_region(37.2,42.01) # NCA/NoCal
    landfall_or = check_region(42.0,46.31) # OR/Oregon
    landfall_wa = check_region(46.3,48.51) # WA/Washington
    return(1, landfall_sca, landfall_nca, landfall_or, landfall_wa, ARmaskcoast)

In [7]:
wusd3 = pd.read_csv('../wusd3_gcms.csv', index_col=0)
for i in range(1011, 1201, 20):
    wusd3.loc[len(wusd3)] = ['cesm2-le', 'CESM2-LE', str(i), '365_day']
wusd3 # The original wusd3_gcms.csv contains just the 15 CMIP6 model data plus WRF-ERA5. Adding the CESM2-LE model data is very simple.

,Model,Model Name,Member,Calendar
0,access-cm2,ACCESS-CM2,r5i1p1f1,standard
1,canesm5,CanESM5,r1i1p2f1,365_day
2,cesm2,CESM2,r11i1p1f1,365_day
3,cnrm-esm2-1,CNRM-ESM2-1,r1i1p1f2,standard
4,ec-earth3,EC-Earth3,r1i1p1f1,standard
5,ec-earth3-veg,EC-Earth3-Veg,r1i1p1f1,standard
6,era5,ERA5,NaN,standard
7,fgoals-g3,FGOALS-g3,r1i1p1f1,365_day
8,giss-e2-1-g,GISS-E2-1-G,r1i1p1f2,365_day
9,miroc6,MIROC6,r1i1p1f1,standard


In [ ]:
for row in range(len(wusd3)):

    model = wusd3.loc[row, 'Model']
    member = wusd3.loc[row, 'Member']
    print(f"Landfall for {model}/{member};")
    
    for ardt in ['G18', 'TE', 'tARget']:
        print(f'{ardt}:', end=" ")
        t1 = time.time()
        ARmask = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARmasks/{ardt}/ARmask.{model}.{member}.nc', use_cftime=True)
        ARmask = ARmask.chunk({'time':'auto', 'lat':-1, 'lon':-1})
        land = xa.open_dataset('/glade/derecho/scratch/tcorrie/regrids/landmask_regridded.nc')
        coast = xa.where(land['LANDMASK'] >= 0.5, 1, 0).idxmax(dim='lon')
        coast_lats = coast.sel(lat=slice(32.5,48.51)).lat.data
        times = ARmask.time.values
        # This is where the parallelization comes in. Map the process_timestep to each time for each ardt/model/member configuration.
        futures = client.map(process_timestep, range(len(times)), ardt=ardt, model=model, member=member)    
        results = client.gather(futures) # This step is responsible for processing the above line. Even with parallelization it may take quite some time.
        
        landfalls_wus, landfalls_sca, landfalls_nca, landfalls_or, landfalls_wa, ARmaskcoasts = zip(*results)

        # Take the overall and regional landfall output and put it into a netcdf file for convenience later use
        landfall_ds = xa.Dataset(
                coords=dict(time=('time',times)),
                data_vars=dict(
                    landfall_wus=(["time"], np.array(landfalls_wus)),
                    landfall_sca=(["time"], np.array(landfalls_sca)),
                    landfall_nca=(["time"], np.array(landfalls_nca)),
                    landfall_or=(["time"], np.array(landfalls_or)),
                    landfall_wa=(["time"], np.array(landfalls_wa)),
                ),
                attrs=dict(description='A binary mask for discerning landfalling ARs over the Western US and split by region')
                
            )
        # Save the coastal AR mask data in a separate netcdf file for easier processing/plotting later on.
        coast_mask_ds = xa.Dataset(
            coords=dict(
                time=('time', times),
                lat=('lat', coast_lats)
            ),
            data_vars=dict(
                ARcoast=(['time','lat'], np.array(ARmaskcoasts))
            ),
            attrs=dict(description='The points where an AR mask overlaps the WUS coast.')
        )
        
        landfall_ds.to_netcdf(f'/glade/work/tcorrie/ARdata/ARlandfall/{ardt}/ARlandfall.{ardt}.{model}.{member}.nc')
        coast_mask_ds.to_netcdf(f'/glade/work/tcorrie/ARdata/ARcoast/{ardt}/ARcoast.{ardt}.{model}.{member}.nc')
        t2 = time.time()
        print(f'{(t2-t1)//60:.0f}\'{(t2-t1)%60:04.1f}\" run time')
        gc.collect()

Landfall for access-cm2/r5i1p1f1;
tARget: 5'54.1" run time
Landfall for canesm5/r1i1p2f1;
tARget: 6'05.7" run time
Landfall for cesm2/r11i1p1f1;
tARget: 5'53.6" run time
Landfall for cnrm-esm2-1/r1i1p1f2;
tARget: 6'10.3" run time
Landfall for ec-earth3/r1i1p1f1;
tARget: 6'07.6" run time
Landfall for ec-earth3-veg/r1i1p1f1;
tARget: 6'09.9" run time
Landfall for era5/nan;
tARget: 3'02.2" run time
Landfall for fgoals-g3/r1i1p1f1;
tARget: 6'03.2" run time
Landfall for giss-e2-1-g/r1i1p1f2;
tARget: 6'03.4" run time
Landfall for miroc6/r1i1p1f1;
tARget: 6'10.3" run time
Landfall for mpi-esm1-2-hr/r3i1p1f1;
tARget: 